In [1]:
# Module 6: Path to Production
# Lab: Session Resumption — Crash Recovery

# Setup -- install dependencies (run once per session)
# !pip install -q claude-agent-sdk python-dotenv

In [1]:
# Import libraries for async agent sessions, file paths, env vars, and SDK API
import os
import json
import asyncio
from pathlib import Path
from dotenv import load_dotenv
from claude_agent_sdk import (
    query,  # Core: sends prompt, yields streaming ResultMessage objects
    ClaudeAgentOptions,  # Configures tools, model, max_turns, resume, permissions
    ResultMessage,  # Terminal message carrying session_id and final result
    get_session_messages,  # Reads full transcript for a session ID
    list_sessions,  # Scans local store and returns all session summaries
)

In [2]:
# Load API keys from .env file
# load_dotenv() reads .env in CWD into os.environ (does not override existing vars)
load_dotenv()
# os.getenv() reads from environment; key set by .env or already exported
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")
print(f"Anthropic key (SDK): {'Yes' if ANTHROPIC_API_KEY else 'No'}")

Anthropic key (SDK): Yes


In [3]:
# Step 4a -- Setup: paths and prompts
# Path.resolve() converts relative "data" to an absolute path like
# /Users/.../Module6/data so the model never misinterprets it as root /data
DATA_DIR = Path("data").resolve()
# TASK is the initial prompt: read state files and continue refactoring
TASK = f"Read {DATA_DIR}/task_state.json and {DATA_DIR}/work_in_progress.txt, then continue the refactoring work described."
# FOLLOW_UP is minimal because the session transcript already has full context
FOLLOW_UP = "Continue exactly where you left off."

In [4]:
# Step 4b -- Crash: run with max_turns=2, capture session_id from ResultMessage
# ClaudeAgentOptions configures tool access, turn limits, and model choice
# max_turns=2 is intentionally low — forces early termination to simulate a crash
options = ClaudeAgentOptions(
    allowed_tools=["Read", "Glob", "Grep", "Edit"],  # Tools for file-based refactoring
    max_turns=2,  # Turn limit: agent stops after 2 tool-use cycles
    model="claude-haiku-4-5-20251001",
)
session_id = None  # Will be populated if we receive a ResultMessage
try:
    # query() returns an async generator — yields streaming messages in real time
    async for message in query(prompt=TASK, options=options):
        # ResultMessage is the terminal message type with session_id + final result
        if isinstance(message, ResultMessage):
            session_id = message.session_id  # Capture the UUID for later recovery
            if message.subtype == "success":
                print(f"[Done] {message.result[:200]}")
except Exception as e:
    # Catches turn-limit errors (expected), network failures, timeouts, etc.
    # session_id is preserved in the outer scope even if the task didn't complete
    print(f"[Crash] {e}")

[Crash] Claude Code returned an error result: Reached maximum number of turns (2)


In [5]:
# Step 4c -- Inspect: read session history after crash
# Only inspect if the crash cell above actually captured a session_id
if session_id:
    # get_session_messages() reads the persisted transcript from disk
    # No agent runs — this is a pure read-only inspection of past activity
    messages = get_session_messages(session_id)
    print(f"\n--- Session {session_id[:8]}... ({len(messages)} messages) ---")
    for i, msg in enumerate(messages):
        role = msg.type.upper()  # 'USER' or 'ASSISTANT'
        # msg.message is a dict with role, content blocks, tool calls, etc.
        # Truncated to 120 chars for a compact readable overview
        preview = str(msg.message)[:120]
        print(f"  [{i}] {role}: {preview}")


--- Session a236e881... (13 messages) ---
  [0] USER: {'role': 'user', 'content': 'Read /Users/ayushsingh/Work/Labs/SDK/Module6/data/task_state.json and /Users/ayushsingh/Wor
  [1] ASSISTANT: {'model': 'claude-haiku-4-5-20251001', 'id': 'msg_011CdXyxKS7uYkTx9ra1m8jp', 'type': 'message', 'role': 'assistant', 'co
  [2] ASSISTANT: {'model': 'claude-haiku-4-5-20251001', 'id': 'msg_011CdXyxKS7uYkTx9ra1m8jp', 'type': 'message', 'role': 'assistant', 'co
  [3] USER: {'role': 'user', 'content': [{'tool_use_id': 'toolu_01RGNc6feADrgdvKXPYRYRDK', 'type': 'tool_result', 'content': '1\t{\n
  [4] ASSISTANT: {'model': 'claude-haiku-4-5-20251001', 'id': 'msg_011CdXyxKS7uYkTx9ra1m8jp', 'type': 'message', 'role': 'assistant', 'co
  [5] USER: {'role': 'user', 'content': [{'tool_use_id': 'toolu_01D1ZSsKcPHirgUoiPJQjsY9', 'type': 'tool_result', 'content': '1\tRef
  [6] ASSISTANT: {'model': 'claude-haiku-4-5-20251001', 'id': 'msg_011CdXyxUKuZDMcXWuhA8sU4', 'type': 'message', 'role': 'assistant', 'co
  [7] 

In [6]:
# Step 4d -- Resume: continue from last state
# resume=session_id loads the full transcript and prepends it to context
# The agent sees everything from before the crash plus the new follow-up prompt
if session_id:
    options = ClaudeAgentOptions(
        allowed_tools=["Read", "Glob", "Grep", "Edit"],
        resume=session_id,  # Key: tells SDK to restore this session's history
        model="claude-haiku-4-5-20251001",
    )
    # FOLLOW_UP is deliberately minimal — session transcript already has full context
    async for message in query(prompt=FOLLOW_UP, options=options):
        if isinstance(message, ResultMessage) and message.subtype == "success":
            print(f"[Resumed] {message.result[:300]}")

[Resumed] Based on the task state and work in progress files I've read, here's what I've identified:

**Current Refactoring Status:**
- **Phase:** JWT Migration (files_identified phase)
- **Changes Completed (1/5):** Replaced session.login() with jwt.login() in src/auth/login.py
- **Next Steps:**
  1. Update 


In [7]:
# Step 5 -- List all available sessions on disk
# list_sessions() scans ~/.claude/projects/<encoded-cwd>/ and returns
# every session summary — useful when you didn't capture the ID at runtime
sessions = list_sessions()
print(f"\n--- All Sessions ({len(sessions)}) ---")
for s in sessions:
    # session_id: UUID for resume= or get_session_messages()
    # first_prompt: first 60 chars of initial user prompt
    # created_at: timestamp as string (epoch ms or ISO 8601)
    print(f"  {s.session_id[:12]}... | {s.first_prompt[:60]} | {s.created_at}")


--- All Sessions (35) ---
  a236e881-dff... | Read /Users/ayushsingh/Work/Labs/SDK/Module6/data/task_state | 1785402671297
  8c4c9067-e0e... | Read /Users/ayushsingh/Work/Labs/SDK/Module6/data/task_state | 1785402370093
  4ac87068-6af... | Read ./data/task_state.json and ./data/work_in_progress.txt, | 1785402170674
  4b2f2b74-8cf... | Read data/task_state.json and data/work_in_progress.txt, the | 1785402097666
  fff297e1-6b1... | Read Module6/data/task_state.json and Module6/data/work_in_p | 1785402001297
  61c0003b-84e... | Read data/task_state.json and data/work_in_progress.txt, the | 1785401868017
  8ae11ced-6af... | Analyze the project at data and update only PATCH and MINOR  | 1785396642247
  985791ef-74c... | Analyze the project at data and update only PATCH and MINOR  | 1785393699670
  5b918b8e-510... | Analyze the project at data and update only PATCH and MINOR  | 1785393580906
  9128044d-eaa... | Analyze the project at data and update only PATCH and MINOR  | 1785393458097
  2